# Relatório 01 — Limpeza e Consolidação da Base de Churn

**Projeto:** Previsão de evasão de clientes (churn) em 60 dias

**Etapa:** preparação dos dados (Aulas 1 e 2 — PBL / CIn-UFPE)

**Entrada:** `data/raw/base_churn_vf2.csv`

**Saída:** `data/processed/base_limpa.csv`

---

### Resumo

Este relatório tenta documentar o caminho que a base de clientes percorre desde o arquivo bruto recebido da operadora até uma tabela pronta para análise exploratória e modelagem. O trabalho começa com um ***diagnóstico de qualidade***, que mede a base nas quatro dimensões formais vistas na Aula 2 (completude, validade, consistência e unicidade) e revela quais problemas existem e em que quantidade. Cada problema encontrado no diagnóstico motiva, em seguida, uma ação de ***tratamento***: remoção de duplicatas, padronização de categorias e neutralização de valores sentinela. Na sequência, a base passa pela **consolidação** proposta na Aula 1, em que informações que estavam em formato de evento (datas soltas, listas de chamados) são transformadas em atributos por cliente, e pela ***construção do rótulo*** `churn`, que define o que o modelo deverá prever. Por fim, uma ***auditoria final*** repete as medições do início e comprova, em números, o efeito de cada intervenção.

Desse modo, partimos de **5.020 linhas** e chegamos a **4.976 clientes** únicos, com **35,4% de churn**, validade, consistência e unicidade em **100%** e um registro (log) de tudo o que foi alterado.

## 1. Introdução

### 1.1 O problema de negócio

Churn é o nome dado à evasão de clientes: o momento em que um assinante encerra o contrato com a operadora. Reter um cliente costuma ser muito mais barato do que conquistar um novo, por isso a empresa quer ***agir antes do cancelamento***, oferecendo condições especiais aos clientes com maior risco. Para isso, precisa de um modelo que, olhando a situação de cada cliente em um dado momento, estime a chance de ele cancelar em um horizonte futuro.

Do ponto de vista técnico, trata-se de um problema de **classificação binária supervisionada** e, como desafio posterior deste projeto, uma não supervisionada: cada cliente recebe o rótulo 1 (cancelou) ou 0 (permaneceu), e o algoritmo aprende, a partir de exemplos passados, a relação entre os atributos do cliente e esse rótulo. Como o custo de não identificar um cliente que vai sair é alto, a métrica prioritária na fase de modelagem será o *recall* da classe 1, mas nada disso funciona se os dados que alimentam o modelo estiverem
errados.

### 1.2 A "fotografia" do cliente: data de corte e janela de desfecho

Um modelo de churn precisa separar com rigor o que é ***passado*** (o que sabemos do cliente) do que é ***futuro*** (o que queremos prever). Para isso fixamos uma ***data de corte***, **21/09/2026**, que funciona como uma fotografia: todos os atributos descrevem o cliente como ele
era nesse dia. A partir dela abre-se uma ***janela de desfecho de 60 dias***, até **20/11/2026**. Se o cliente cancelou dentro dessa janela, seu rótulo é 1; caso contrário, é 0.

```
   passado (atributos)                      futuro (rótulo)
───────────────────────────────┃──────────────────────────────┃───▶
                         21/09/2026                     20/11/2026
                       data de corte                 fim da janela (60 dias)
```

Essa separação tem uma consequência importante: **nenhuma informação posterior à data de corte pode entrar nos atributos**. Se entrasse, o modelo estaria "vendo o futuro" no treino, um tipo de vazamento de dados (*data leakage*) que produz métricas excelentes no laboratório e um modelo inútil em produção.

### 1.3 Granularidade: uma linha por cliente

A Aula 1 chama atenção para a **granularidade** dos dados, isto é, o que cada linha da tabela representa. Nosso problema é tomar uma decisão por cliente, portanto a unidade de análise é o **cliente na data de corte**, e a tabela
final deve ter exatamente uma linha por cliente. A base recebida, porém, ainda guarda informações em nível de **evento**. Portanto, consolidar significa resumir esses eventos em números por cliente, sem perder o que eles dizem.

### 1.4 O que entra (e o que não entra) neste notebook

Uma regra guia todas as decisões a seguir: aqui só aplicamos correções ***determinísticas***, ou seja, que não dependem de estatísticas aprendidas da base.

Já a **imputação** de ausentes, o **escalonamento** e a **codificação one-hot** dependem de parâmetros calculados a partir dos dados. Pelo protocolo de prevenção de vazamento da Aula 2, esses parâmetros devem ser aprendidos (`.fit`) somente no conjunto de treino e apenas aplicados (`.transform`) no teste. Essas transformações ficam, portanto, para o notebook 03, depois da divisão feita no notebook 02.

### 1.5 Roteiro do relatório

O texto segue a ordem natural do trabalho. A **Seção 2** carrega a base e faz o primeiro contato com ela. A **Seção 3** mede a qualidade nas quatro dimensões e investiga o mecanismo das ausências. A **Seção 4** trata os problemas encontrados no diagnóstico. A **Seção 5** consolida os eventos em atributos por cliente. A **Seção 6** constrói o rótulo `churn` e define a
população que será modelada. A **Seção 7** valida o resultado, compara o antes e o depois e exporta a base. A **Seção 8** conclui e aponta o que fica para as próximas etapas.

Ao longo do caminho mantemos dois registros que, no fim, contam a história do processo: o `log_correcoes`, que anota cada alteração e quantas linhas ela afetou, e o `funil`, que acompanha quantas linhas a base tem depois de cada etapa.

### Preparação do ambiente

A célula abaixo deixa o notebook executável tanto no Google Colab quanto localmente. No Colab,
ela clona o repositório do projeto e instala as dependências; localmente, apenas sobe para a
pasta raiz, onde está o `config.py`. Esse arquivo centraliza as constantes do projeto (caminhos,
datas, listas de colunas, dicionários de-para e faixas válidas). Mantê-las em um único lugar
garante que todos os notebooks usem exatamente as mesmas regras.

In [1]:
import os, sys

# Se estiver rodando no Google Colab, baixa o repositório do projeto, entra na pasta,
# atualiza o código e instala as bibliotecas listadas em requirements.txt.
if "google.colab" in sys.modules:
    if not os.path.exists("/content/previsao_de_evasao_de_clientes"):
        !git clone -q https://github.com/Gagdm/previsao_de_evasao_de_clientes.git /content/previsao_de_evasao_de_clientes
    %cd /content/previsao_de_evasao_de_clientes
    !git pull -q
    !pip install -q -r requirements.txt
# Se estiver rodando localmente a partir da pasta notebooks/, sobe um nível para a raiz do
# projeto, de modo que "import config" e os caminhos relativos funcionem.
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..

import numpy as np
import pandas as pd
import config         # constantes do projeto: caminhos, datas de corte, de-para, faixas válidas

/content/previsao_de_evasao_de_clientes


## 2. Leitura e primeiro contato com a base

O primeiro passo é carregar a base **exatamente como ela chegou**, sem nenhuma correção. Isso pode parecer óbvio, mas é uma escolha metodológica: o diagnóstico da Seção 3 precisa enxergar os problemas originais, e a auditoria final da Seção 7 precisa de um "antes" intacto para comparar com o "depois". Por isso, o DataFrame lido aqui, `df_bruto`, nunca será modificado; todas as
correções serão feitas em uma cópia.

Há um único ajuste na leitura, que não altera o conteúdo: a coluna de mensalidade usa vírgula como separador decimal. Informamos isso ao pandas para que ela seja lida como número, e não como texto. As colunas de data, por outro lado, ficam propositalmente como texto: queremos que a auditoria veja os formatos originais, inclusive os que estão fora do padrão.

In [2]:
# Lê o arquivo CSV bruto. O parâmetro decimal="," faz o pandas interpretar "69,79" como 69.79,
# de forma que mensalidade_rs já chegue como número (float) e possa ser usada em contas.
df_bruto = pd.read_csv(config.BASE_BRUTA, decimal=",")

# .shape devolve (n_linhas, n_colunas): é o tamanho da base antes de qualquer tratamento.
print(f"Linhas: {df_bruto.shape[0]} | Colunas: {df_bruto.shape[1]}")

# Mostra as 5 primeiras linhas para conferir visualmente se a leitura deu certo.
df_bruto.head()

Linhas: 5020 | Colunas: 10


/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,id_cliente,data_aquisicao_cliente,mensalidade_rs,data_chamado,atrasos_6m,tipo_contrato,forma_pagamento,fibra_optica,debito_automatico,data_ocorrencia_churn
0,C0001,21/04/2024,67.96,"09/09/2026, 15/08/2026",0.0,Bienal,Boleto,0,0,NaN
1,C0002,21/02/2025,65.95,NaN,0.0,Mensal,Cartão,0,0,NaN
2,C0003,21/03/2025,100.88,28/08/2026,1.0,Anual,Boleto,1,1,NaN
3,C0004,21/03/2025,94.80,"06/08/2026, 28/07/2026",2.0,Mensal,Cartão,0,1,NaN
4,C0005,21/02/2022,53.68,05/07/2026,2.0,Anual,Pix,0,0,NaN


In [3]:
# .info() lista cada coluna com seu tipo (dtype) e a quantidade de valores não vazios.
# É o primeiro diagnóstico rápido: colunas com "Non-Null Count" menor que o total de linhas
# têm valores ausentes, e colunas do tipo "object" estão sendo lidas como texto.
df_bruto.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5020 entries, 0 to 5019
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id_cliente              5020 non-null   object 
 1   data_aquisicao_cliente  5020 non-null   object 
 2   mensalidade_rs          4900 non-null   float64
 3   data_chamado            3967 non-null   object 
 4   atrasos_6m              4985 non-null   float64
 5   tipo_contrato           4994 non-null   object 
 6   forma_pagamento         5000 non-null   object 
 7   fibra_optica            5020 non-null   int64  
 8   debito_automatico       5020 non-null   int64  
 9   data_ocorrencia_churn   1789 non-null   object 
dtypes: float64(2), int64(2), object(6)
memory usage: 392.3+ KB


O primeiro contato já antecipa boa parte do que será investigado. A base tem **5.020 linhas e
10 colunas**. Quatro colunas obrigatórias têm lacunas (`mensalidade_rs`, `atrasos_6m`,
`tipo_contrato` e `forma_pagamento`). As duas colunas de data de evento, `data_chamado` e
`data_ocorrencia_churn`, também têm muitos vazios, mas, como veremos, nelas o vazio tem
significado. Além disso, três colunas que deveriam ser datas aparecem como texto (`object`), e
`data_chamado` mostra mais de uma data na mesma célula, sinal de que a base ainda não está na
granularidade de uma linha por cliente.

Para transformar essas impressões em medidas objetivas, passamos à auditoria.

## 3. Diagnóstico de qualidade (auditoria inicial)

Auditar a base significa compará-la com um **contrato de dados**: o conjunto de regras que descreve como cada atributo deveria ser. Na Aula 2, esse contrato aparece como o *inventário de atributos e dicionário de validação*, que registra, para cada coluna, o tipo esperado, a faixa
ou o domínio aceito e se ela é obrigatória. No projeto, essas regras estão codificadas em
`config.py`.

A qualidade é medida em **quatro dimensões complementares**. Cada uma responde a uma pergunta diferente, e um dado pode estar perfeito em uma dimensão e falhar em outra:

| Dimensão | Pergunta que responde | Exemplo de falha nesta base |
| --- | --- | --- |
| **Completude** | O dado está preenchido? | mensalidade em branco |
| **Validade** | O valor respeita o tipo, a faixa e o domínio definidos? | `'cartao'` em vez de `'Cartão'`; mensalidade de R$ 9.999 |
| **Consistência** | As colunas concordam entre si? | cancelamento datado antes da aquisição |
| **Unicidade** | Cada entidade aparece uma única vez? | o mesmo cliente em duas linhas |

Nesta seção **nada é corrigido**; apenas medimos. Ao final, cada dimensão recebe um percentual de conformidade, que servirá de linha de base para a comparação da Seção 7.

### 3.1 Completude

A completude mede a proporção de células preenchidas. Um detalhe importante é decidir **quais
colunas entram na conta**. Nem todo vazio é falha: em `data_chamado`, uma célula vazia quer dizer
que o cliente nunca abriu chamado; em `data_ocorrencia_churn`, quer dizer que o cliente não
cancelou. Nesses casos, o vazio é uma informação legítima e penalizá-lo distorceria a métrica.
Por isso a completude é calculada apenas sobre as **colunas obrigatórias**, aquelas em que todo
cliente deveria ter um valor.

In [4]:
# Colunas que, pelo dicionário de dados, todo cliente deveria ter preenchidas.
# data_chamado e data_ocorrencia_churn ficam de fora porque nelas o vazio tem significado.
colunas_obrigatorias = ["id_cliente", "data_aquisicao_cliente", "mensalidade_rs", "atrasos_6m",
                        "tipo_contrato", "forma_pagamento", "fibra_optica", "debito_automatico"]

completude = pd.DataFrame({
    # .isna() cria uma máscara com True onde a célula está vazia; .sum() conta os True por coluna.
    "ausentes": df_bruto[colunas_obrigatorias].isna().sum(),
    # .notna().mean() é a proporção de células preenchidas por coluna; * 100 converte em %.
    "% preenchido": (df_bruto[colunas_obrigatorias].notna().mean() * 100).round(2),
})

# .values transforma a tabela em uma matriz única; a média sobre ela considera todas as células
# juntas (e não coluna a coluna). O resultado é a completude geral da base.
pct_completude = df_bruto[colunas_obrigatorias].notna().values.mean() * 100
print(f"Completude geral: {pct_completude:.1f}%")
completude

Completude geral: 99.5%


,ausentes,% preenchido
id_cliente,0,100.00
data_aquisicao_cliente,0,100.00
mensalidade_rs,120,97.61
atrasos_6m,35,99.30
tipo_contrato,26,99.48
forma_pagamento,20,99.60
fibra_optica,0,100.00
debito_automatico,0,100.00


A completude geral é alta (**99,5%**), mas ela não está distribuída por igual. A coluna mais
afetada é `mensalidade_rs`, com 120 ausências (2,4% das linhas); `atrasos_6m`, `tipo_contrato` e
`forma_pagamento` têm lacunas pequenas. Essas ausências **não serão preenchidas aqui**: como
discutido na introdução, qualquer preenchimento estatístico precisa ser aprendido só no treino.
O que podemos fazer agora é entender **por que** os dados faltam, o que é feito na Seção 3.5.

### 3.2 Validade

A validade verifica se cada valor preenchido pertence ao conjunto permitido para sua coluna. Para atributos categóricos, isso significa pertencer ao ***domínio categórico***, a lista fechada de valores oficiais para o tipo de contrato. Um valor com o mesmo significado mas grafado de outra forma é
chamado de ***valor não canônico***: o humano entende, mas o computador trata como uma categoria diferente. Para atributos numéricos, a regra é uma ***faixa aceitável*** (mensalidade entre
R\$ 10 e R\$ 500; atrasos entre 0 e 6, já que a janela é de seis meses). Para as binárias, só 0 ou 1. Para as datas, um formato fixo `dd/mm/aaaa`.

O código monta uma tabela de violações em que cada coluna é uma regra e cada célula indica se aquela linha quebra aquela regra. Uma linha é considerada inválida se quebrar **pelo menos uma** regra.

In [5]:
# Domínio canônico de cada categórica = os valores de destino do dicionário de-para.
canonicos_contrato = set(config.DE_PARA["tipo_contrato"].values())     # {Mensal, Anual, Bienal}
canonicos_pagamento = set(config.DE_PARA["forma_pagamento"].values())  # {Boleto, Cartão, Pix}

# Limites mínimos e máximos aceitos para as numéricas, também definidos no config.
min_mensalidade, max_mensalidade = config.FAIXAS_VALIDAS["mensalidade_rs"]
min_atrasos, max_atrasos = config.FAIXAS_VALIDAS["atrasos_6m"]

# Cada coluna desta tabela é uma regra; True significa que a linha viola aquela regra.
# Os testes começam com .notna() para que ausências não sejam contadas como inválidas
# (ausência já foi medida na completude; aqui só avaliamos valores que existem).
violacoes = pd.DataFrame({
    # Valor preenchido que não está no domínio canônico do tipo de contrato.
    "contrato_nao_canonico": df_bruto["tipo_contrato"].notna()
        & ~df_bruto["tipo_contrato"].isin(canonicos_contrato),
    # Valor preenchido que não está no domínio canônico da forma de pagamento.
    "pagamento_nao_canonico": df_bruto["forma_pagamento"].notna()
        & ~df_bruto["forma_pagamento"].isin(canonicos_pagamento),
    # Mensalidade preenchida fora da faixa [10, 500]. .between inclui os dois limites.
    "mensalidade_fora_da_faixa": df_bruto["mensalidade_rs"].notna()
        & ~df_bruto["mensalidade_rs"].between(min_mensalidade, max_mensalidade),
    # Quantidade de atrasos preenchida fora da faixa [0, 6].
    "atrasos_fora_da_faixa": df_bruto["atrasos_6m"].notna()
        & ~df_bruto["atrasos_6m"].between(min_atrasos, max_atrasos),
    # Alguma das colunas binárias com valor diferente de 0 ou 1.
    "binaria_invalida": ~df_bruto["fibra_optica"].isin([0, 1])
        | ~df_bruto["debito_automatico"].isin([0, 1]),
    # Data de aquisição que não segue o padrão dd/mm/aaaa (expressão regular: 2 dígitos,
    # barra, 2 dígitos, barra, 4 dígitos, do início ao fim do texto).
    "data_fora_do_formato": ~df_bruto["data_aquisicao_cliente"].str.match(r"^\d{2}/\d{2}/\d{4}$"),
})

# .any(axis=1) olha cada linha e devolve True se ela violar pelo menos uma regra.
linhas_invalidas = violacoes.any(axis=1)
pct_validade = (1 - linhas_invalidas.mean()) * 100
print(f"Linhas com alguma violação: {linhas_invalidas.sum()} | Validade: {pct_validade:.1f}%")

# Soma de cada coluna = quantas linhas violam cada regra.
violacoes.sum()

Linhas com alguma violação: 203 | Validade: 96.0%


,0
contrato_nao_canonico,80
pagamento_nao_canonico,90
mensalidade_fora_da_faixa,12
atrasos_fora_da_faixa,8
binaria_invalida,0
data_fora_do_formato,15


Saber **quantas** violações existem não basta; para decidir como tratá-las é preciso ver
**quais** são os valores problemáticos. A célula seguinte filtra, para cada regra, apenas as
linhas que a violam e conta os valores encontrados.

In [6]:
# Para cada regra, usamos a coluna correspondente de "violacoes" como filtro (.loc[máscara, coluna])
# e contamos os valores que aparecem. map(repr) mostra o texto entre aspas, deixando visíveis
# espaços extras (ex.: 'bienal ' com espaço no fim).
print(df_bruto.loc[violacoes["contrato_nao_canonico"], "tipo_contrato"].map(repr).value_counts(), "\n")
print(df_bruto.loc[violacoes["pagamento_nao_canonico"], "forma_pagamento"].value_counts(), "\n")
print(df_bruto.loc[violacoes["mensalidade_fora_da_faixa"], "mensalidade_rs"].value_counts(), "\n")
print(df_bruto.loc[violacoes["atrasos_fora_da_faixa"], "atrasos_6m"].value_counts(), "\n")
# Para as datas, basta ver alguns exemplos para entender o formato alternativo.
print(df_bruto.loc[violacoes["data_fora_do_formato"], "data_aquisicao_cliente"].head())

tipo_contrato
'mensal'     40
'anual'      25
'bienal '    15
Name: count, dtype: int64 

forma_pagamento
cartao    50
boleto    25
PIX       15
Name: count, dtype: int64 

mensalidade_rs
-85.0      8
 9999.0    4
Name: count, dtype: int64 

atrasos_6m
99.0    8
Name: count, dtype: int64 

88     2024-08-21
386    2026-06-21
550    2022-09-21
700    2021-12-21
890    2024-12-21
Name: data_aquisicao_cliente, dtype: object


A validade inicial é de **96,0%** (203 linhas com alguma violação), e o detalhamento revela três famílias de problemas, cada uma com uma causa e um tratamento diferentes:

***Categorias não canônicas (170 casos).*** `tipo_contrato` aparece como `'mensal'`, `'anual'` e `'bienal '` (minúsculas e, no último caso, com um espaço no fim); `forma_pagamento` aparece como `'cartao'` (sem acento), `'boleto'` e `'PIX'`. São típicos de digitação livre ou de sistemas
legados que gravam o mesmo conceito de formas diferentes. Se não forem corrigidos, um algoritmo enxergaria seis tipos de contrato em vez de três e dividiria artificialmente o padrão de comportamento de cada grupo. O tratamento é um **dicionário de-para** (Seção 4.2).

***Valores numéricos impossíveis (20 casos).*** As mensalidades fora da faixa são exatamente −85 e 9.999, e os atrasos fora da faixa são exatamente 99. A repetição de valores idênticos e redondos é a assinatura de **valores sentinela**: códigos usados por algum sistema para indicar "desconhecido". Não são clientes extremos, e sim ausências disfarçadas (Seção 4.3).

***Datas em formato alternativo (15 casos).*** Algumas datas de aquisição estão no padrão ISO `aaaa-mm-dd`. A informação está correta, só a escrita é diferente; basta convertê-las com um leitor que aceite os dois formatos (Seção 5.1).

### 3.3 Consistência

Enquanto a validade avalia cada coluna isoladamente, a consistência avalia **regras que envolvem mais de uma coluna**. Um valor pode ser válido sozinho e ainda assim contradizer outro. Nesta base há duas regras temporais óbvias: um cliente **não pode cancelar** antes de ter sido adquirido, e **não pode abrir chamado** antes de ser cliente.

A segunda regra exige um passo a mais, porque `data_chamado` guarda várias datas na mesma célula, separadas por vírgula. A técnica usada é "explodir" essa lista: cada data vira uma linha própria, mas mantendo o índice (o número da linha) do cliente de origem. Assim é possível comparar cada chamado com a data de aquisição do respectivo cliente e, depois, reagrupar por cliente. As datas são convertidas apenas para esta verificação; `df_bruto` continua intacto.

In [7]:
# Converte as datas de aquisição. format="mixed" aceita formatos diferentes linha a linha
# (dd/mm/aaaa e aaaa-mm-dd), e dayfirst=True garante que 05/07 seja lido como 5 de julho.
aquisicao = pd.to_datetime(df_bruto["data_aquisicao_cliente"], format="mixed", dayfirst=True)
# Data de cancelamento; células vazias viram NaT (o "NaN" das datas).
cancelamento = pd.to_datetime(df_bruto["data_ocorrencia_churn"], format="%d/%m/%Y")

# Regra 1: cancelamento anterior à aquisição. Comparações com NaT resultam em False,
# então quem não cancelou nunca é marcado como inconsistente.
churn_antes_da_aquisicao = cancelamento < aquisicao

# Regra 2: chamado anterior à aquisição.
# .str.split(", ") transforma "09/09/2026, 15/08/2026" na lista ["09/09/2026", "15/08/2026"];
# .explode() cria uma linha por elemento da lista, repetindo o índice do cliente de origem.
chamados = pd.to_datetime(df_bruto["data_chamado"].str.split(", ").explode(), format="%d/%m/%Y")
chamado_antes_da_aquisicao = (
    # Compara cada chamado com a aquisição do mesmo cliente (alinhado pelo índice).
    (chamados < aquisicao.loc[chamados.index])
    # Reagrupa por cliente (level=0 é o índice original): True se ALGUM chamado for anterior.
    .groupby(level=0).any()
    # Garante uma entrada para todos os clientes, inclusive quem não tem chamado (False).
    .reindex(df_bruto.index, fill_value=False)
)

inconsistencias = pd.DataFrame({
    "churn_antes_da_aquisicao": churn_antes_da_aquisicao,
    "chamado_antes_da_aquisicao": chamado_antes_da_aquisicao,
})

# Uma linha é inconsistente se violar qualquer uma das duas regras.
linhas_inconsistentes = inconsistencias.any(axis=1)
pct_consistencia = (1 - linhas_inconsistentes.mean()) * 100
print(f"Linhas inconsistentes: {linhas_inconsistentes.sum()} | Consistência: {pct_consistencia:.1f}%")
print(inconsistencias.sum(), "\n")

# Mostra os clientes com cancelamento antes da aquisição, para inspeção manual.
df_bruto.loc[churn_antes_da_aquisicao, ["id_cliente", "data_aquisicao_cliente", "data_ocorrencia_churn"]]

Linhas inconsistentes: 43 | Consistência: 99.1%
churn_antes_da_aquisicao      10
chamado_antes_da_aquisicao    33
dtype: int64 



,id_cliente,data_aquisicao_cliente,data_ocorrencia_churn
202,C0203,21/08/2024,15/01/2020
494,C0495,21/08/2025,15/01/2020
1356,C1357,21/12/2024,15/01/2020
1572,C1573,21/10/2021,15/01/2020
2026,C2027,21/12/2024,15/01/2020
2722,C2723,21/09/2024,15/01/2020
2999,C3000,21/03/2026,15/01/2020
3522,C3523,21/06/2026,15/01/2020
3644,C3645,21/05/2026,15/01/2020
4914,C4915,21/10/2025,15/01/2020


A consistência inicial é de **99,1%**, com *43 linhas problemáticas*. Os dois tipos de inconsistência pedem tratamentos diferentes, e a diferença está em **qual informação está comprometida**:

- Os ***10 cancelamentos anteriores à aquisição*** têm todos a mesma data, 15/01/2020, o que sugere:

  Um valor padrão gravado por erro de sistema. Como essa data é justamente a origem do rótulo `churn`, não sabemos se o cliente cancelou ou não. Um rótulo errado ensina ao modelo o padrão inverso do verdadeiro (ruído de rótulo, Aula 2), e não há como reconstruí-lo com segurança. Esses clientes serão **removidos** na Seção 6.
- Os ***33 clientes com algum chamado anterior à aquisição*** têm um problema mais localizado:

  Apenas algumas datas de chamado estão erradas, enquanto o restante do registro é aproveitável. A decisão é **descartar apenas os chamados impossíveis** na contagem (Seção 5.2), preservando o cliente.

### 3.4 Unicidade

A unicidade garante que cada entidade apareça uma só vez na tabela. Pela definição de granularidade, a chave primária é o cliente na data de corte (`id_cliente` + data de corte); como esta base contém uma única fotografia, basta verificar `id_cliente`.

Há dois cenários de duplicidade, com gravidades distintas. Se as linhas repetidas forem **idênticas**, a solução é trivial: manter uma cópia. Se tiverem o **mesmo ID com conteúdo diferente**, há um conflito, e seria preciso uma regra para decidir qual versão é a correta. O código distingue os dois casos.

In [8]:
# .duplicated() marca como True cada ocorrência repetida (a primeira aparição fica como False).
ids_repetidos = df_bruto["id_cliente"].duplicated()   # mesmo ID, independentemente do conteúdo
linhas_identicas = df_bruto.duplicated()              # linha inteira igual a uma anterior

pct_unicidade = (1 - ids_repetidos.mean()) * 100
print(f"IDs repetidos: {ids_repetidos.sum()}")
print(f"Linhas idênticas a outra: {linhas_identicas.sum()}")
# ID repetido mas linha diferente = conflito de informação, o caso mais grave.
print(f"Repetidos com conteúdo diferente: {(ids_repetidos & ~linhas_identicas).sum()}")
print(f"Unicidade: {pct_unicidade:.1f}%")

IDs repetidos: 20
Linhas idênticas a outra: 20
Repetidos com conteúdo diferente: 0
Unicidade: 99.6%


Há **20 IDs repetidos**, e todos correspondem a linhas inteiramente idênticas. Não existe,
portanto, nenhum conflito a resolver: são cópias provavelmente geradas por uma carga duplicada,
e basta manter uma de cada (Seção 4.1). A unicidade inicial é de **99,6%**.

### 3.5 Por que a mensalidade está faltando? (MCAR, MAR ou MNAR)

Antes de decidir como tratar ausências, é preciso entender o **mecanismo** que as gerou. A estatística distingue três situações:

- **MCAR** (*Missing Completely at Random*): a falta é puramente acidental e não tem relação com nenhuma variável, observada ou não. Exemplo: uma falha momentânea no sistema de faturamento. Neste caso, a ausência não introduz viés e qualquer imputação razoável funciona.
- **MAR** (*Missing at Random*): a probabilidade de faltar depende de **outras colunas observadas**. Exemplo: clientes de um determinado plano têm o campo em branco por protocolo do setor. Neste caso, a imputação deve considerar essas outras colunas (por exemplo, mediana por grupo ou KNNImputer).
- **MNAR** (*Missing Not at Random*): a probabilidade de faltar depende do **próprio valor ausente** ou do fenômeno que estudamos. Exemplo: clientes insatisfeitos com o preço deixam de atualizar o cadastro. Aqui a ausência é, ela mesma, uma informação; a recomendação da Aula 2 é criar um **indicador binário de ausência** (`mensalidade_ausente`) além da imputação, para que o modelo possa aproveitar esse sinal.

Não é possível provar o mecanismo apenas olhando os dados, mas é possível reunir evidências. Fazemos dois testes simples:

**(1)** se a taxa de ausência muda conforme uma variável observada (aqui, `fibra_optica`), há indício de MAR;

**(2)** se a taxa de churn é diferente entre quem tem e quem não tem mensalidade, a ausência carrega informação sobre o alvo, o que aponta para MNAR.

In [9]:
# Máscara: True para clientes sem mensalidade registrada.
mensalidade_ausente = df_bruto["mensalidade_rs"].isna()
# Aproximação do rótulo apenas para este teste: True se há qualquer data de cancelamento.
cancelou = df_bruto["data_ocorrencia_churn"].notna()

# Teste 1 (MAR?): a proporção de ausentes muda entre clientes com e sem fibra óptica?
# groupby(...).mean() sobre uma máscara booleana dá a proporção de True em cada grupo.
print("% de mensalidade ausente por fibra óptica:")
print(mensalidade_ausente.groupby(df_bruto["fibra_optica"]).mean().mul(100).round(1), "\n")

# Teste 2 (MNAR?): quem não tem mensalidade cancela em proporção diferente de quem tem?
print("% que cancelou, com e sem mensalidade:")
print(cancelou.groupby(mensalidade_ausente).mean().mul(100).round(1))

% de mensalidade ausente por fibra óptica:
fibra_optica
0    2.4
1    2.3
Name: mensalidade_rs, dtype: float64 

% que cancelou, com e sem mensalidade:
mensalidade_rs
False    35.5
True     40.0
Name: data_ocorrencia_churn, dtype: float64


O primeiro teste não mostra relação com `fibra_optica`: a taxa de ausência é praticamente igual nos dois grupos (2,4% e 2,3%), o que enfraquece a hipótese de MAR para essa variável. O segundo teste mostra uma diferença: **40,0%** de cancelamento entre os clientes sem mensalidade contra **35,5%** entre os demais.

Essa diferença deve ser lida com cautela. Com apenas 120 clientes sem mensalidade, uma diferença de 4,5 pontos percentuais está dentro do que a variação aleatória poderia produzir, ou seja, não há evidência forte de MNAR, mas também não podemos descartá-la. Como criar o indicador de ausência
é barato e não causa vazamento, a decisão registrada para o notebook 03 é **imputar a mensalidade e criar a variável `mensalidade_ausente`**, deixando que a modelagem diga se ela é útil.

### 3.6 Síntese do diagnóstico

A tabela abaixo reúne as quatro medições. Ela é o "retrato do antes" e será comparada com o resultado final na Seção 7.

In [10]:
# Consolida as quatro métricas em uma tabela de resumo. Ela fica guardada em "resumo_antes"
# e será reaproveitada na auditoria final para a comparação antes x depois.
resumo_antes = pd.DataFrame({
    "dimensão": ["Completude", "Validade", "Consistência", "Unicidade"],
    "% conformidade": [pct_completude, pct_validade, pct_consistencia, pct_unicidade],
    "o que mede": ["células obrigatórias preenchidas",
                   "linhas sem valor fora das regras",
                   "linhas sem contradição entre colunas",
                   "linhas com id_cliente único"],
}).round(1)
resumo_antes

,dimensão,% conformidade,o que mede
0,Completude,99.5,células obrigatórias preenchidas
1,Validade,96.0,linhas sem valor fora das regras
2,Consistência,99.1,linhas sem contradição entre colunas
3,Unicidade,99.6,linhas com id_cliente único


À primeira vista todas as dimensões parecem altas, acima de 96%. Mas a leitura correta não é "a base está quase boa": cada ponto percentual que falta corresponde a problemas concretos que, se ignorados, produziriam efeitos desproporcionais. Quatro mensalidades de R$ 9.999 bastam
para inflar a média e o desvio padrão da coluna inteira; dez rótulos errados ensinam ao modelo exemplos falsos; vinte linhas duplicadas dão peso dobrado a alguns clientes. A seção seguinte trata esses problemas, um a um, na ordem em que fazem sentido.

In [11]:
df = df_bruto.copy()
log_correcoes = []

linhas_antes = len(df)
ids_duplicados = df.loc[df.duplicated(), "id_cliente"].tolist()

df = df.drop_duplicates().reset_index(drop=True)

removidas = linhas_antes - len(df)
log_correcoes.append({
    "etapa": "3. Duplicatas",
    "descricao": "linhas idênticas removidas",
    "linhas_afetadas": removidas,
})

print(f"Antes: {linhas_antes} | Depois: {len(df)} | Removidas: {removidas}")
print(f"id_cliente é único? {df['id_cliente'].is_unique}")
print(f"IDs que estavam duplicados: {ids_duplicados}")

Antes: 5020 | Depois: 5000 | Removidas: 20
id_cliente é único? True
IDs que estavam duplicados: ['C2388', 'C4576', 'C1456', 'C0807', 'C0497', 'C1268', 'C3601', 'C4374', 'C1151', 'C3976', 'C2163', 'C1114', 'C3386', 'C0857', 'C2033', 'C3618', 'C1643', 'C3711', 'C1235', 'C4645']


## 4. Tratamento dos problemas encontrados

A partir daqui as correções passam a ser aplicadas. Todas são feitas em `df`, uma cópia da base bruta, e cada uma é anotada no `log_correcoes` com a etapa, a descrição e o número de linhas afetadas. Esse registro é o que permite, no fim, explicar exatamente o que aconteceu com os dados, uma exigência de reprodutibilidade e também de governança, já que a Aula 2 recomenda
preservar o valor original em log sempre que um dado é alterado.

Também começamos o `funil`, que registra o tamanho da base após cada etapa.

### 4.1 Remoção de duplicatas

A auditoria mostrou que as 20 repetições são cópias idênticas, sem conflito de conteúdo. A remoção vem primeiro por um motivo prático: se as duplicatas permanecessem, todas as contagens seguintes estariam infladas, e o log ficaria impreciso.

In [12]:
# Cópia de trabalho: todas as correções acontecem em df, e df_bruto fica intacto para a
# comparação final. Sem .copy(), alterações em df poderiam afetar df_bruto.
df = df_bruto.copy()

# Registro de todas as correções: cada item é um dicionário com etapa, descrição e linhas afetadas.
log_correcoes = []
# Registro do tamanho da base ao fim de cada etapa (o "funil" do processo).
funil = [{"etapa": "Base bruta", "linhas": len(df)}]

linhas_antes = len(df)
# Guarda os IDs que estavam duplicados, para documentação.
ids_duplicados = df.loc[df.duplicated(), "id_cliente"].tolist()

# drop_duplicates() remove linhas inteiramente iguais a uma anterior, mantendo a primeira.
# reset_index(drop=True) renumera o índice de 0 a n-1, sem buracos.
df = df.drop_duplicates().reset_index(drop=True)

removidas = linhas_antes - len(df)
log_correcoes.append({
    "etapa": "4.1 Duplicatas",
    "descricao": "linhas idênticas removidas",
    "linhas_afetadas": removidas,
})
funil.append({"etapa": "4.1 Sem duplicatas", "linhas": len(df)})

print(f"Antes: {linhas_antes} | Depois: {len(df)} | Removidas: {removidas}")
# .is_unique confirma que agora não há nenhum ID repetido.
print(f"id_cliente é único? {df['id_cliente'].is_unique}")
print(f"IDs que estavam duplicados: {ids_duplicados}")

Antes: 5020 | Depois: 5000 | Removidas: 20
id_cliente é único? True
IDs que estavam duplicados: ['C2388', 'C4576', 'C1456', 'C0807', 'C0497', 'C1268', 'C3601', 'C4374', 'C1151', 'C3976', 'C2163', 'C1114', 'C3386', 'C0857', 'C2033', 'C3618', 'C1643', 'C3711', 'C1235', 'C4645']


A base passa de 5.020 para **5.000 linhas**, e `id_cliente` torna-se uma chave única. A partir deste ponto, cada linha corresponde a um cliente distinto.

### 4.2 Padronização das categorias (de-para)

As 170 grafias não canônicas encontradas na auditoria são corrigidas em duas etapas. Primeiro, uma ***normalização*** genérica do texto: remover espaços nas pontas (`strip`) e converter para minúsculas (`lower`). Isso já reduz muitas variações a uma forma comum, `'PIX'` e `'Pix'` viram `'pix'`, `'bienal '` vira `'bienal'`. Depois, um **dicionário de-para** traduz cada forma normalizada para o valor canônico oficial (`'pix'` → `'Pix'`, `'cartao'` → `'Cartão'`).

Um cuidado importante: se surgir uma grafia que o dicionário não conhece, o código **interrompe a execução** com um erro, em vez de transformá-la silenciosamente em vazio. Esse é o comportamento de "bloqueio de pipeline" discutido no contrato de dados da Aula 2: é preferível parar e revisar o dicionário a perder informação sem perceber. Valores que já eram vazios
continuam vazios; eles serão imputados no notebook 03.

In [13]:
trocas = []   # guarda o detalhamento "de -> para" de cada coluna, para documentação

# config.DE_PARA tem uma entrada por coluna categórica: {coluna: {grafia_normalizada: canônico}}.
for coluna, de_para in config.DE_PARA.items():
    original = df[coluna]
    # Passo 1 - normalização: remove espaços das pontas e converte para minúsculas.
    normalizado = original.str.strip().str.lower()
    # Passo 2 - de-para: .map(dicionário) troca cada valor pelo canônico.
    # Valores que não estão no dicionário (e os vazios) viram NaN.
    padronizado = normalizado.map(de_para)

    # Trava de segurança: valor preenchido que virou NaN = grafia sem tradução no dicionário.
    # Nesse caso a execução para, para que o dicionário seja revisado.
    sem_mapeamento = normalizado.notna() & padronizado.isna()
    if sem_mapeamento.any():
        raise ValueError(f"Valores sem de-para em {coluna}: {normalizado[sem_mapeamento].unique()}")

    # Linhas que de fato mudaram (estavam preenchidas e ficaram diferentes do original).
    alterado = original.notna() & (original != padronizado)
    trocas.append(pd.DataFrame({
        "coluna": coluna,
        "de": original[alterado].map(repr),   # repr deixa espaços extras visíveis
        "para": padronizado[alterado],
    }))
    log_correcoes.append({
        "etapa": "4.2 De-para",
        "descricao": f"{coluna} padronizado",
        "linhas_afetadas": int(alterado.sum()),
    })

    # Substitui a coluna original pela versão padronizada.
    df[coluna] = padronizado

# Junta as trocas das duas colunas e conta quantas linhas tiveram cada substituição.
tabela_trocas = pd.concat(trocas).value_counts().reset_index(name="linhas")
tabela_trocas

,coluna,de,para,linhas
0,forma_pagamento,'cartao',Cartão,50
1,tipo_contrato,'mensal',Mensal,40
2,tipo_contrato,'anual',Anual,25
3,forma_pagamento,'boleto',Boleto,25
4,forma_pagamento,'PIX',Pix,15
5,tipo_contrato,'bienal ',Bienal,15


In [14]:
# Conferência: depois do de-para, cada coluna deve ter apenas os três valores canônicos
# (e os vazios originais, que aparecem como NaN graças a dropna=False).
for coluna in config.DE_PARA:
    print(df[coluna].value_counts(dropna=False), "\n")

tipo_contrato
Mensal    2757
Anual     1201
Bienal    1017
NaN         25
Name: count, dtype: int64 

forma_pagamento
Boleto    1968
Cartão    1739
Pix       1273
NaN         20
Name: count, dtype: int64 



As 170 substituições aconteceram como esperado (80 em `tipo_contrato` e 90 em
`forma_pagamento`), e cada coluna agora tem exatamente as três categorias oficiais. Repare que o grupo `Mensal` passou a ter 2.757 clientes: se a correção não tivesse sido feita, 40 desses clientes estariam em uma categoria separada (`'mensal'`), e o modelo aprenderia sobre eles como se fossem um tipo de contrato diferente. Os 25 e 20 vazios restantes são ausências legítimas que ficam para a imputação.

### 4.3 Neutralização dos valores sentinela

Um **valor sentinela** é um número escolhido por um sistema para significar "não sei" ou "não se aplica". Ele é perigoso justamente porque **parece um dado**: está preenchido, é numérico e passa em qualquer verificação de completude. Na auditoria, −85 e 9.999 em `mensalidade_rs` e 99 em `atrasos_6m` foram identificados como sentinelas por três motivos: são valores impossíveis (não existe mensalidade negativa nem 99 atrasos em seis meses), repetem-se de forma idêntica e são números "redondos" típicos de código de sistema.

Se ficassem na base, distorceriam qualquer estatística: uma única mensalidade de R\$ 9.999 entre valores típicos de R\$ 50 a R\$ 140 é suficiente para deslocar a média e inflar muito o desvio padrão, o que afetaria a imputação, o escalonamento e modelos sensíveis a escala. O tratamento correto **não é** remover o cliente nem "cortar" o valor no limite da faixa (isso inventaria uma
mensalidade), e sim reconhecer o que o sentinela realmente é: um valor ausente. Por isso eles são convertidos para `NaN` e passam a ser tratados junto com as demais ausências no notebook 03.

In [15]:
# config.SENTINELAS = {coluna: [lista de códigos que significam "desconhecido"]}
for coluna, sentinelas in config.SENTINELAS.items():
    # Máscara com True nas linhas cujo valor está na lista de sentinelas daquela coluna.
    eh_sentinela = df[coluna].isin(sentinelas)
    # Mostra quantas vezes cada sentinela apareceu, antes de substituí-lo.
    print(f"{coluna}: {df.loc[eh_sentinela, coluna].value_counts().to_dict()}")

    log_correcoes.append({
        "etapa": "4.3 Sentinelas",
        "descricao": f"{coluna}: {sentinelas} -> NA",
        "linhas_afetadas": int(eh_sentinela.sum()),
    })
    # Substitui o sentinela por NaN (ausente). O cliente continua na base.
    df.loc[eh_sentinela, coluna] = np.nan

# Verificação: depois da troca, nenhum valor preenchido pode estar fora da faixa válida.
# assert interrompe a execução se a condição for falsa; aqui funciona como teste automático.
for coluna, (minimo, maximo) in config.FAIXAS_VALIDAS.items():
    fora_da_faixa = df[coluna].notna() & ~df[coluna].between(minimo, maximo)
    assert not fora_da_faixa.any(), f"{coluna} ainda tem {fora_da_faixa.sum()} valores fora de [{minimo}, {maximo}]"

print("\nNenhum valor fora da faixa válida.")

mensalidade_rs: {-85.0: 8, 9999.0: 4}
atrasos_6m: {99.0: 8}

Nenhum valor fora da faixa válida.


In [16]:
# Comparação das estatísticas descritivas antes (base bruta) e depois (sentinelas como NaN).
# describe() mostra contagem, média, desvio padrão, mínimo, quartis e máximo.
colunas = list(config.SENTINELAS)
print("ANTES (df_bruto):")
print(df_bruto[colunas].describe().round(2), "\n")
print("DEPOIS (df):")
print(df[colunas].describe().round(2))

ANTES (df_bruto):
       mensalidade_rs  atrasos_6m
count         4900.00     4985.00
mean            91.24        1.20
std            283.90        4.04
min            -85.00        0.00
25%             66.98        0.00
50%             85.92        1.00
75%             98.48        2.00
max           9999.00       99.00 

DEPOIS (df):
       mensalidade_rs  atrasos_6m
count         4868.00     4957.00
mean            83.41        1.05
std             18.37        0.95
min             49.90        0.00
25%             67.08        0.00
50%             85.94        1.00
75%             98.48        2.00
max            136.42        5.00


A comparação mostra de forma concreta por que os sentinelas precisavam sair. Em `mensalidade_rs`, o **desvio padrão cai de 283,90 para 18,37** e a média passa de R\$ 91,24 para R\$ 83,41; o mínimo deixa de ser −85 e o máximo deixa de ser 9.999, passando a uma faixa realista (R\$ 49,90 a R\$ 136,42). Em `atrasos_6m`, o desvio cai de 4,04 para 0,95 e o máximo passa a 5.
Repare também que a **mediana praticamente não muda** (85,92 → 85,94): a mediana é uma medida robusta a valores extremos, enquanto a média e o desvio padrão não são. É exatamente, por isso, que a mediana será a escolha natural para a imputação no notebook 03.

Com os sentinelas convertidos, a coluna `mensalidade_rs` passa de 120 para 132 ausências, e `atrasos_6m` de 35 para 43 (já descontadas as duplicatas). A completude "piora" um pouco, mas essa piora é honesta: esses valores nunca foram conhecidos.

## 5. Consolidação: de eventos para atributos do cliente

Até aqui corrigimos valores. Agora mudamos a **forma** da informação para que ela fique na granularidade definida na introdução: uma linha por cliente, com atributos numéricos que um algoritmo consegue usar. Duas colunas precisam dessa transformação: `data_chamado`, que guarda uma lista de eventos, e `data_aquisicao_cliente`, que guarda uma data. Nenhum algoritmo de
classificação usa uma data ou uma lista diretamente; o que interessa ao modelo é ***quanto*** e ***há quanto tempo***.

Em ambos os casos vale a regra temporal da Seção 1.2: os atributos só podem usar informações disponíveis ***até a data de corte***.

### 5.1 Padronização das datas de aquisição

Primeiro, convertemos definitivamente a data de aquisição de texto para o tipo data. O leitor com `format="mixed"` resolve, ao mesmo tempo, os 15 registros em formato ISO identificados na validade. Essa conversão é pré-requisito tanto para filtrar os chamados quanto para calcular o tempo de casa.

### 5.2 Contagem de chamados (`chamados_totais`)

Cada célula de `data_chamado` contém zero, uma ou várias datas. Resumimos isso em uma única variável, `chamados_totais`, que conta **quantos chamados válidos** o cliente abriu. Um chamado é considerado válido se ocorreu **depois da aquisição** (chamados anteriores são as inconsistências da Seção 3.3) e **até a data de corte** (chamados posteriores seriam informação
do futuro). Uma célula vazia significa zero chamados, não é um dado ausente, e, por isso, essa variável não terá nenhum `NaN`.

O número de chamados é um indicador clássico de insatisfação: clientes que precisam acionar o suporte com frequência tendem a ter problemas recorrentes com o serviço.

In [17]:
# 5.1 - Conversão definitiva da data de aquisição para datetime.
# format="mixed" aceita dd/mm/aaaa e aaaa-mm-dd; dayfirst=True resolve a ambiguidade dia/mês.
df["data_aquisicao_cliente"] = pd.to_datetime(df["data_aquisicao_cliente"], format="mixed", dayfirst=True)
# Se alguma data não pudesse ser interpretada, ela viraria NaT. Esperamos 0.
print(f"Datas de aquisição não convertidas: {df['data_aquisicao_cliente'].isna().sum()}")

# 5.2 - Uma linha por chamado: separa a lista de datas e "explode" em linhas,
# preservando no índice o número da linha do cliente de origem.
chamados = pd.to_datetime(df["data_chamado"].str.split(", ").explode(), format="%d/%m/%Y")
# Para cada chamado, busca a data de aquisição do cliente correspondente (alinhado pelo índice).
aquisicao = df.loc[chamados.index, "data_aquisicao_cliente"]

# Classifica cada chamado:
antes_da_aquisicao = chamados < aquisicao            # impossível -> descartado
depois_do_corte = chamados > config.DATA_CORTE       # informação do futuro -> descartado
# Válido = existe (não é NaT de célula vazia) e não caiu em nenhum dos descartes acima.
valido = chamados.notna() & ~antes_da_aquisicao & ~depois_do_corte

# Soma os chamados válidos por cliente (True conta 1). Clientes sem chamado somam 0.
df["chamados_totais"] = valido.groupby(level=0).sum()

log_correcoes.append({
    "etapa": "5.2 Chamados",
    "descricao": "chamados anteriores à aquisição descartados",
    # Conta clientes (e não datas) com pelo menos um chamado descartado.
    "linhas_afetadas": int(antes_da_aquisicao.groupby(level=0).any().sum()),
})

print(f"Datas de chamado na base: {chamados.notna().sum()}")
print(f"Descartadas — antes da aquisição: {antes_da_aquisicao.sum()} | depois do corte: {depois_do_corte.sum()}")
# Mostra lado a lado a coluna original e a contagem, para conferência visual.
df[["data_chamado", "chamados_totais"]].head()

Datas de aquisição não convertidas: 0
Datas de chamado na base: 8597
Descartadas — antes da aquisição: 52 | depois do corte: 0


,data_chamado,chamados_totais
0,"09/09/2026, 15/08/2026",2
1,NaN,0
2,28/08/2026,1
3,"06/08/2026, 28/07/2026",2
4,05/07/2026,1


In [18]:
# Distribuição da nova variável: quantos clientes têm 0, 1, 2, ... chamados.
df["chamados_totais"].value_counts().sort_index()

,count
chamados_totais,
0,1065
1,1470
2,1222
3,671
4,352
5,151
6,46
7,12
8,7


Das 8.597 datas de chamado registradas, **52 foram descartadas** por serem anteriores à aquisição (distribuídas entre 33 clientes, exatamente os apontados na auditoria de consistência); nenhuma era posterior à data de corte. A distribuição resultante é a esperada para uma contagem de eventos: concentrada em valores baixos (1.065 clientes sem chamado e 1.470 com um) e com uma cauda longa à direita que chega a 10 chamados. Esses poucos clientes com muitos chamados não são erros, são justamente os casos que mais interessam ao modelo.

### 5.3 Tempo de casa (`tempo_cliente_meses`)

A data de aquisição, por si só, não significa nada para um algoritmo; o que importa é ***há quanto tempo*** a pessoa é cliente na data da fotografia. Calculamos, portanto, o número de **meses completos** entre a aquisição e a data de corte. O cálculo é feito em duas partes: primeiro a diferença de meses pelo calendário (anos × 12 + meses); depois, se o dia da aquisição
ainda não foi "alcançado" no mês do corte, subtrai-se 1, porque aquele mês ainda não se completou. Por exemplo, um cliente adquirido em 25/08/2026 tem 0 meses completos em 21/09/2026, não 1.

O tempo de casa é um dos preditores mais importantes em churn: clientes recentes ainda não criaram vínculo com o serviço e costumam cancelar muito mais que os antigos.

In [19]:
aquisicao = df["data_aquisicao_cliente"]
corte = config.DATA_CORTE   # 21/09/2026

# Verificação de sanidade: ninguém pode ter sido adquirido depois da data da fotografia.
assert (aquisicao <= corte).all(), "Há clientes adquiridos depois da data de corte"

# Diferença de meses pelo calendário: (diferença de anos * 12) + (diferença de meses).
meses = (corte.year - aquisicao.dt.year) * 12 + (corte.month - aquisicao.dt.month)
# Se o dia da aquisição é maior que o dia do corte, o último mês ainda não está completo: -1.
# (aquisicao.dt.day > corte.day) é True/False; .astype(int) transforma em 1/0.
meses = meses - (aquisicao.dt.day > corte.day).astype(int)

df["tempo_cliente_meses"] = meses

# Resumo estatístico da nova variável e algumas linhas para conferência.
print(df["tempo_cliente_meses"].describe().round(1))
df[["data_aquisicao_cliente", "tempo_cliente_meses"]].head()

count    5000.0
mean       24.7
std        15.5
min         1.0
25%        13.0
50%        21.0
75%        33.0
max        72.0
Name: tempo_cliente_meses, dtype: float64


,data_aquisicao_cliente,tempo_cliente_meses
0,2024-04-21,29
1,2025-02-21,19
2,2025-03-21,18
3,2025-03-21,18
4,2022-02-21,55


O tempo de casa varia de ***1 a 72 meses***, com mediana de 21 meses: a carteira é composta em sua maioria por clientes com um a três anos de relacionamento. O mínimo de 1 mês já sinaliza algo que será tratado na Seção 6.2: existem clientes que acabaram de entrar e cujo histórico é curto demais.

Com as duas variáveis derivadas criadas, a base já está na granularidade correta. Falta definir o que o modelo vai prever.

## 6. Rótulo e definição da população

### 6.1 Construção do rótulo `churn`

O rótulo é a resposta que o modelo deve aprender a prever. Ele é derivado da data de cancelamento, comparada com três marcos: a data de aquisição, a data de corte (21/09/2026) e o fim da janela de desfecho (20/11/2026). Cada cliente cai em exatamente uma das situações abaixo:

| Situação | O que significa | Decisão |
| --- | --- | --- |
| Não cancelou | data de cancelamento vazia | `churn = 0` |
| Cancelou na janela | entre 22/09 e 20/11/2026 | `churn = 1` |
| Cancelou depois da janela | após 20/11/2026: na fotografia, ainda é um cliente ativo e não cancelou no horizonte | `churn = 0` |
| Cancelou antes do corte | já não era cliente na data da fotografia | removido |
| Cancelou antes da aquisição | data impossível (erro de registro) | removido |

As duas últimas situações saem da base por motivos diferentes. Quem cancelou **antes do corte** não faz parte da população: na data da fotografia, essa pessoa já não era cliente, e não faz sentido prever se ela vai cancelar. Quem tem cancelamento **antes da aquisição** tem um rótulo corrompido: não sabemos qual seria a resposta correta, e manter o registro com um palpite introduziria ruído justamente na variável que o modelo usa como verdade.

A ordem das condições no código importa: elas são avaliadas de cima para baixo e o cliente recebe a primeira que se aplica. Por isso "antes da aquisição" é testado antes de "antes do corte", uma data de 2020 também é anterior ao corte, mas o diagnóstico correto é o de erro. Antes de aplicar a regra, listamos os clientes com cancelamento impossível (são os mesmos 10
identificados na auditoria).

In [20]:
# Converte a data de cancelamento só para inspecionar os casos impossíveis antes de removê-los.
cancelamento_bruto = pd.to_datetime(df["data_ocorrencia_churn"], format="%d/%m/%Y")
df.loc[cancelamento_bruto < df["data_aquisicao_cliente"],
       ["id_cliente", "data_aquisicao_cliente", "data_ocorrencia_churn"]]

,id_cliente,data_aquisicao_cliente,data_ocorrencia_churn
202,C0203,2024-08-21,15/01/2020
494,C0495,2025-08-21,15/01/2020
1356,C1357,2024-12-21,15/01/2020
1572,C1573,2021-10-21,15/01/2020
2026,C2027,2024-12-21,15/01/2020
2722,C2723,2024-09-21,15/01/2020
2999,C3000,2026-03-21,15/01/2020
3522,C3523,2026-06-21,15/01/2020
3644,C3645,2026-05-21,15/01/2020
4914,C4915,2025-10-21,15/01/2020


In [21]:
cancelamento = pd.to_datetime(df["data_ocorrencia_churn"], format="%d/%m/%Y")

# np.select avalia as condições em ordem e atribui o rótulo da PRIMEIRA condição verdadeira.
# Quem não satisfaz nenhuma recebe o default ("depois_da_janela").
situacao = pd.Series(np.select(
    [
        cancelamento.isna(),                            # sem data -> não cancelou
        cancelamento < df["data_aquisicao_cliente"],    # data impossível
        cancelamento <= config.DATA_CORTE,              # já tinha saído na data da fotografia
        cancelamento <= config.FIM_JANELA,              # cancelou dentro dos 60 dias
    ],
    ["nao_cancelou", "antes_da_aquisicao", "antes_do_corte", "na_janela"],
    default="depois_da_janela",
), index=df.index)

# Quantos clientes em cada situação.
print(situacao.value_counts(), "\n")

# Clientes que saem da base: rótulo corrompido ou fora da população.
invalidos = situacao.isin(["antes_da_aquisicao", "antes_do_corte"])
log_correcoes.append({
    "etapa": "6.1 Rótulo churn",
    "descricao": "cancelamento antes da aquisição ou do corte: cliente removido",
    "linhas_afetadas": int(invalidos.sum()),
})

# O rótulo: 1 só para quem cancelou dentro da janela; 0 para todos os demais que permanecem.
# (situacao == "na_janela") é True/False; .astype(int) converte para 1/0.
df["churn"] = (situacao == "na_janela").astype(int)
# Remove os clientes inválidos (~ inverte a máscara) e renumera o índice.
df = df.loc[~invalidos].reset_index(drop=True)
funil.append({"etapa": "6.1 Rótulo válido", "linhas": len(df)})

# Contrato do alvo (Aula 2): churn só pode conter 0 ou 1, sem ausentes. Se falhar, bloqueia.
assert df["churn"].isin([0, 1]).all(), "churn tem valores diferentes de 0 e 1"
print(f"Clientes: {len(df)} | Churn: {df['churn'].sum()} ({df['churn'].mean():.1%})")

nao_cancelou          3218
na_janela             1772
antes_da_aquisicao      10
Name: count, dtype: int64 

Clientes: 4990 | Churn: 1772 (35.5%)


Dos 5.000 clientes, 3.218 não cancelaram, 1.772 cancelaram dentro da janela e **10 foram removidos** por terem cancelamento anterior à aquisição. Não houve nenhum caso de cancelamento antes do corte nem depois da janela, o que indica que a base já foi extraída considerando apenas clientes ativos em 21/09/2026 e desfechos observados até 20/11/2026.

A taxa de churn resultante é de **35,5%**. É um desbalanceamento moderado, a classe positiva é minoritária, mas não rara, que precisará ser considerado na divisão treino/teste (estratificação) e na escolha das métricas, mas não exige, por si só, técnicas agressivas de balanceamento.

### 6.2 Recorte por maturação: clientes com menos de 2 meses de casa

A Aula 1 levanta uma pergunta delicada: se queremos prever churn nos próximos 60 dias, faz sentido incluir um cliente que entrou há apenas 30 dias? A resposta é não, e o motivo é o **viés de janela de observação**. Para que o modelo aprenda uma relação confiável entre o comportamento passado e o desfecho, todos os clientes precisam ter um histórico observável comparável. Um cliente com um mês de casa tem `atrasos_6m` calculado sobre poucas semanas (e não seis meses), `chamados_totais` acumulado em um período mínimo e um comportamento típico de quem ainda está experimentando o serviço, algo qualitativamente diferente de um cliente estabelecido.

Misturar os dois grupos confundiria o algoritmo. A abordagem recomendada é usar um **critério de maturação**: o modelo principal considera apenas clientes com tempo de casa de pelo menos o tamanho do horizonte de previsão (aqui, 2 meses). Os clientes imaturos não são simplesmente descartados: eles ficam registrados em `clientes_imaturos` e poderiam ser objeto de um modelo
específico de *churn precoce*.

In [22]:
# Máscara de clientes com tempo de casa menor que o mínimo definido no config (2 meses).
imaturo = df["tempo_cliente_meses"] < config.MATURACAO_MINIMA_MESES

# Guarda esses clientes à parte (.copy() evita que dependam de df) para documentação
# e eventual uso em um modelo de churn precoce.
clientes_imaturos = df.loc[imaturo, ["id_cliente", "data_aquisicao_cliente",
                                     "tempo_cliente_meses", "churn"]].copy()

log_correcoes.append({
    "etapa": "6.2 Maturação",
    "descricao": f"clientes com menos de {config.MATURACAO_MINIMA_MESES} meses de casa removidos",
    "linhas_afetadas": int(imaturo.sum()),
})

# Mantém apenas os clientes maduros na base principal.
df = df.loc[~imaturo].reset_index(drop=True)
funil.append({"etapa": "6.2 Clientes maduros", "linhas": len(df)})

print(f"Removidos: {len(clientes_imaturos)} | Churn entre eles: {clientes_imaturos['churn'].mean():.0%}")
print(f"Clientes restantes: {len(df)} | Churn: {df['churn'].sum()} ({df['churn'].mean():.1%})")
clientes_imaturos

Removidos: 14 | Churn entre eles: 64%
Clientes restantes: 4976 | Churn: 1763 (35.4%)


,id_cliente,data_aquisicao_cliente,tempo_cliente_meses,churn
157,C0158,2026-08-21,1,1
985,C0988,2026-08-21,1,0
1673,C1678,2026-08-21,1,1
1743,C1748,2026-08-21,1,1
1908,C1913,2026-08-21,1,1
2211,C2217,2026-08-21,1,1
2228,C2234,2026-08-21,1,0
2316,C2322,2026-08-21,1,0
2323,C2329,2026-08-21,1,1
3421,C3429,2026-08-21,1,1


Foram separados ***14 clientes***, todos adquiridos em 21/08/2026 (um mês antes do corte). A taxa de churn entre eles é de ***64%***, quase o dobro da média da base. Esse número confirma o argumento teórico: clientes recém-chegados têm uma dinâmica de evasão própria e, se fossem mantidos, o modelo poderia aprender uma associação exagerada entre "pouco tempo de casa" e cancelamento, válida para esse grupo específico mas não para a carteira estabelecida. A base
principal fica com ***4.976 clientes*** e churn de ***35,4%***.

## 7. Validação final e entrega

### 7.1 Seleção das colunas finais

As colunas de data cumpriram seu papel e agora saem da base. `data_aquisicao_cliente` foi resumida em `tempo_cliente_meses`; `data_chamado`, em `chamados_totais`; e `data_ocorrencia_churn`, em `churn`. Esta última **precisa** sair: ela é a própria resposta que o modelo deve prever, e mantê-la seria o caso mais extremo de vazamento, o modelo
"aprenderia" a olhar a data de cancelamento em vez de aprender o comportamento do cliente.

A ordem das colunas segue a organização do `config.py`: identificador, numéricas,categóricas, binárias e, por último, o alvo.

In [24]:
# Monta a lista final na ordem: ID, numéricas, categóricas, binárias e alvo.
# O * "desempacota" cada lista do config dentro da lista final.
colunas_finais = [config.ID, *config.NUMERICAS, *config.CATEGORICAS, *config.BINARIAS, config.ALVO]

# Seleciona apenas essas colunas; .copy() cria uma tabela independente de df.
base_limpa = df[colunas_finais].copy()

print(base_limpa.shape)
base_limpa.head()

(4976, 10)


,id_cliente,tempo_cliente_meses,mensalidade_rs,chamados_totais,atrasos_6m,tipo_contrato,forma_pagamento,fibra_optica,debito_automatico,churn
0,C0001,29,67.96,2,0.0,Bienal,Boleto,0,0,0
1,C0002,19,65.95,0,0.0,Mensal,Cartão,0,0,0
2,C0003,18,100.88,1,1.0,Anual,Boleto,1,1,0
3,C0004,18,94.80,2,2.0,Mensal,Cartão,0,1,0
4,C0005,55,53.68,1,2.0,Anual,Pix,0,0,0


### 7.2 Auditoria final: antes e depois

Para comprovar o efeito do tratamento, repetimos as medições da Seção 3 sobre a base final. As regras são as mesmas, agora aplicadas também às variáveis criadas (incluindo o alvo `churn`, que pelo contrato de dados só pode ser 0 ou 1). A consistência é verificada em `df`, que ainda contém as datas originais: nenhum cliente restante pode ter cancelamento anterior à aquisição, e nenhum chamado contado pode ser anterior à aquisição.

In [25]:
# Colunas de atributos (sem o alvo), sobre as quais medimos a completude.
colunas_atributos = [config.ID, *config.NUMERICAS, *config.CATEGORICAS, *config.BINARIAS]

# --- Completude: proporção de células preenchidas na base final.
pct_completude_depois = base_limpa[colunas_atributos].notna().values.mean() * 100

# --- Validade: as mesmas regras da auditoria inicial, agora na base limpa.
canonicos = {coluna: set(de_para.values()) for coluna, de_para in config.DE_PARA.items()}
violacoes_depois = pd.DataFrame({
    # Uma regra de domínio por coluna categórica (criadas com dict comprehension).
    **{f"{c}_nao_canonico": base_limpa[c].notna() & ~base_limpa[c].isin(v)
       for c, v in canonicos.items()},
    # Uma regra de faixa por coluna numérica com faixa definida.
    **{f"{c}_fora_da_faixa": base_limpa[c].notna() & ~base_limpa[c].between(minimo, maximo)
       for c, (minimo, maximo) in config.FAIXAS_VALIDAS.items()},
    "binaria_invalida": ~base_limpa[config.BINARIAS].isin([0, 1]).all(axis=1),
    "churn_invalido": ~base_limpa[config.ALVO].isin([0, 1]),
})
pct_validade_depois = (1 - violacoes_depois.any(axis=1).mean()) * 100

# --- Consistência: verificada em df, que ainda tem as datas originais.
cancelamento_final = pd.to_datetime(df["data_ocorrencia_churn"], format="%d/%m/%Y")
churn_inconsistente = cancelamento_final < df["data_aquisicao_cliente"]
# Os chamados anteriores à aquisição foram descartados na contagem; logo, chamados_totais
# nunca pode ser maior que o número bruto de datas da célula.
datas_brutas = df["data_chamado"].str.split(", ").str.len().fillna(0)
chamados_inconsistentes = df["chamados_totais"] > datas_brutas
pct_consistencia_depois = (1 - (churn_inconsistente | chamados_inconsistentes).mean()) * 100

# --- Unicidade: proporção de IDs não repetidos.
pct_unicidade_depois = (1 - base_limpa[config.ID].duplicated().mean()) * 100

comparacao = pd.DataFrame({
    "dimensão": resumo_antes["dimensão"],
    "% antes": resumo_antes["% conformidade"],
    "% depois": [pct_completude_depois, pct_validade_depois,
                 pct_consistencia_depois, pct_unicidade_depois],
}).round(1)
comparacao

,dimensão,% antes,% depois
0,Completude,99.5,99.5
1,Validade,96.0,100.0
2,Consistência,99.1,100.0
3,Unicidade,99.6,100.0


In [26]:
# Ausências que permanecem na base final, por coluna. Elas são esperadas e serão
# imputadas no notebook 03, com parâmetros aprendidos apenas no treino.
base_limpa[colunas_atributos].isna().sum()

,0
id_cliente,0
tempo_cliente_meses,0
mensalidade_rs,132
chamados_totais,0
atrasos_6m,43
tipo_contrato,25
forma_pagamento,20
fibra_optica,0
debito_automatico,0


**Validade, consistência e unicidade chegaram a 100%.** Todas as categorias estão no domínio canônico, todos os números estão nas faixas válidas, nenhum cliente tem datas contraditórias e cada ID aparece uma única vez.

A **completude permanece em 99,5%**, e isso é intencional, não uma falha do processo. O número fica estável porque dois efeitos se compensam: os sentinelas convertidos acrescentaram ausências, enquanto as variáveis derivadas (`tempo_cliente_meses`, `chamados_totais`) entraram sem nenhum
vazio. As 132 mensalidades, 43 contagens de atrasos, 25 tipos de contrato e 20 formas de pagamento ausentes são lacunas reais que só podem ser preenchidas com estatísticas da base, e por isso aguardam a divisão treino/teste.

### 7.3 Registro das correções e funil da base

O log reúne, em uma única tabela, todas as intervenções feitas, na ordem em que aconteceram. O funil mostra quantas linhas restaram ao fim de cada etapa que removeu registros. Juntos, eles contam a história completa do processo e permitem que qualquer pessoa reproduza ou questione cada decisão.

In [27]:
# Converte a lista de dicionários do log em uma tabela.
tabela_log = pd.DataFrame(log_correcoes)
tabela_log

,etapa,descricao,linhas_afetadas
0,4.1 Duplicatas,linhas idênticas removidas,20
1,4.2 De-para,tipo_contrato padronizado,80
2,4.2 De-para,forma_pagamento padronizado,90
3,4.3 Sentinelas,"mensalidade_rs: [-85, 9999] -> NA",12
4,4.3 Sentinelas,atrasos_6m: [99] -> NA,8
5,5.2 Chamados,chamados anteriores à aquisição descartados,33
6,6.1 Rótulo churn,cancelamento antes da aquisição ou do corte: c...,10
7,6.2 Maturação,clientes com menos de 2 meses de casa removidos,14


In [28]:
# Funil: linhas restantes após cada etapa e quantas saíram em relação à etapa anterior.
tabela_funil = pd.DataFrame(funil)
# .diff() calcula a diferença para a linha anterior; o sinal é invertido para mostrar removidas.
tabela_funil["removidas"] = (-tabela_funil["linhas"].diff()).fillna(0).astype(int)
# Percentual da base bruta que permanece em cada etapa.
tabela_funil["% da base bruta"] = (tabela_funil["linhas"] / tabela_funil["linhas"].iloc[0] * 100).round(1)
tabela_funil

,etapa,linhas,removidas,% da base bruta
0,Base bruta,5020,0,100.0
1,4.1 Sem duplicatas,5000,20,99.6
2,6.1 Rótulo válido,4990,10,99.4
3,6.2 Clientes maduros,4976,14,99.1


O funil deixa claro que a limpeza foi **conservadora**: das 5.020 linhas originais, 99,1% permanecem na base final. Apenas 44 linhas saíram, 20 cópias duplicadas, 10 clientes com rótulo corrompido e 14 clientes imaturos, e cada remoção tem uma justificativa registrada. Todos os demais problemas (categorias, sentinelas, datas em outro formato, chamados impossíveis) foram
corrigidos **sem descartar o cliente**, preservando o máximo de informação para o modelo.

### 7.4 Exportação

A base limpa é salva em `data/processed/base_limpa.csv`. Logo depois, o arquivo é lido de volta
para confirmar que a gravação preservou o número de linhas, de colunas e a taxa de churn. Esse
passo simples evita surpresas no notebook seguinte.

In [29]:
# Cria a pasta data/processed caso ainda não exista (parents=True cria pastas intermediárias;
# exist_ok=True evita erro se ela já existir).
config.PROCESSADOS.mkdir(parents=True, exist_ok=True)
# Salva sem o índice numérico do pandas (index=False), que não é uma informação do cliente.
base_limpa.to_csv(config.BASE_LIMPA, index=False)

# Conferência: relê o arquivo salvo e confirma as dimensões e a taxa de churn.
conferencia = pd.read_csv(config.BASE_LIMPA)
print(f"Salvo em {config.BASE_LIMPA.name}: {conferencia.shape[0]} linhas, {conferencia.shape[1]} colunas")
print(f"Churn: {conferencia['churn'].mean():.1%}")

Salvo em base_limpa.csv: 4976 linhas, 10 colunas
Churn: 35.4%


## 8. Conclusão

Este relatório percorreu o caminho completo entre a base bruta e uma base pronta para a
modelagem, e cada etapa foi motivada pela anterior. O **diagnóstico** mediu a base nas quatro
dimensões de qualidade e mostrou que, apesar de percentuais altos, havia problemas concretos:
20 linhas duplicadas, 170 categorias grafadas fora do padrão, 20 valores sentinela, 15 datas em
formato alternativo, 10 rótulos impossíveis e 52 chamados anteriores à aquisição. A
investigação das ausências indicou um possível sinal na falta de mensalidade, fraco demais para
uma conclusão, mas suficiente para justificar um indicador de ausência na próxima etapa.

O **tratamento** atacou cada problema com a ferramenta adequada: duplicatas idênticas foram
removidas; categorias foram traduzidas por um dicionário de-para que bloqueia a execução diante
de grafias desconhecidas; sentinelas foram reconhecidos como ausências disfarçadas, o que fez o
desvio padrão da mensalidade cair de 283,90 para 18,37. A **consolidação** levou a base à
granularidade correta, transformando listas de datas em `chamados_totais` e datas de aquisição em
`tempo_cliente_meses`, sempre respeitando a data de corte. A **construção do rótulo** separou com
rigor passado e futuro, e o **recorte por maturação** retirou os 14 clientes cujo histórico era
curto demais para o horizonte de 60 dias.

O resultado é uma base com **4.976 clientes, 10 colunas e 35,4% de churn**, com validade,
consistência e unicidade em 100% e todas as alterações documentadas no log.

### O que fica para as próximas etapas

| Notebook | Pendência | Por que não foi feito aqui |
| --- | --- | --- |
| 02 | Divisão treino/teste estratificada pelo `churn` | É o ponto a partir do qual estatísticas podem ser aprendidas |
| 03 | Imputação de `mensalidade_rs` (mediana) e criação de `mensalidade_ausente` | Usa a mediana, que deve vir só do treino |
| 03 | Imputação de `atrasos_6m`, `tipo_contrato` e `forma_pagamento` | Usa mediana/moda do treino |
| 03 | Escalonamento das numéricas e one-hot das categóricas | Usa médias, desvios e categorias aprendidos no treino |
| — | Eventual modelo de churn precoce com `clientes_imaturos` | Problema diferente, fora do escopo do modelo principal |

---
### Anexo — baixar o arquivo gerado (Colab)

Ao rodar no Colab, a célula abaixo baixa a base limpa para o computador. Depois, ela pode ser
enviada para `data/processed/` no GitHub (*Add file → Upload files*).

In [30]:
# Só executa no Colab: dispara o download do CSV gerado para o computador do usuário.
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(config.BASE_LIMPA))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>